# Trabalhando com Várias Fontes de Dados e Arquivos

Notebook interativo para aprender a importar, processar e manipular dados de diferentes origens.

> Este notebook acompanha o guia teórico `fontes_de_dados.md`. Use as células abaixo para executar os exemplos e praticar.


## Setup

Instalação das dependências necessárias. Execute esta célula uma vez antes de começar.


In [ ]:
import sys
!{sys.executable} -m pip install openpyxl pdfplumber PyPDF2 reportlab python-docx -q
print("Dependências instaladas!")


In [ ]:
import pandas as pd
import numpy as np
import json
import csv
import sqlite3
import requests
import xml.etree.ElementTree as ET

print("Bibliotecas carregadas!")


## 1. Por que existem tantos formatos de arquivo?

Cada formato de arquivo existe para resolver um problema diferente:

| Formato | Para que serve |
|---------|---------------|
| **TXT** | Logs, anotações simples, textos corridos |
| **CSV** | Tabelas simples, troca de dados entre sistemas |
| **JSON** | Dados hierárquicos, APIs, configurações |
| **Excel (.xlsx)** | Planilhas com múltiplas abas, formatação visual |
| **XML** | Configurações, padrões corporativos |
| **PDF** | Relatórios finalizados, layout preservado |
| **DOCX** | Documentos editáveis, relatórios formatados |
| **SQLite** | Dados relacionais, consultas SQL |
| **API + JSON** | Dados em tempo real, integração externa |

Saber escolher o formato certo é tão importante quanto saber programar a leitura e escrita dele.


## 2. Arquivos de texto simples (.txt)

Usamos a função `open()` com um **modo de abertura**:
- `'w'` — escrita (cria/sobrescreve)
- `'r'` — leitura
- `'a'` — append (adiciona ao final)

**Sempre** use `with open(...)` — o arquivo é fechado automaticamente, mesmo em caso de erro.


In [ ]:
# Escrevendo em um arquivo
with open('exemplo.txt', 'w', encoding='utf-8') as f:
    f.write('Primeira linha\n')
    f.write('Segunda linha\n')
    f.write('Terceira linha\n')

print('Arquivo criado!')


In [ ]:
# Lendo o arquivo inteiro
with open('exemplo.txt', 'r', encoding='utf-8') as f:
    conteudo = f.read()

print(conteudo)


In [ ]:
# Lendo linha por linha (eficiente para arquivos grandes)
with open('exemplo.txt', 'r', encoding='utf-8') as f:
    for linha in f:
        print(linha.strip())


In [ ]:
# Adicionando conteúdo sem apagar o que existe (modo 'a')
with open('exemplo.txt', 'a', encoding='utf-8') as f:
    f.write('Quarta linha adicionada depois\n')

with open('exemplo.txt', 'r', encoding='utf-8') as f:
    print(f.read())


## 3. CSV (Comma-Separated Values)

CSV é o formato de "planilha em texto puro". Vamos ver duas abordagens:

1. **Módulo `csv`** — já vem com Python, bom para entender o funcionamento
2. **Pandas** — padrão da indústria, transforma CSV em DataFrame


In [ ]:
# Criando um CSV com o módulo csv
dados = [
    ['nome', 'idade', 'cidade'],
    ['Ana', 28, 'São Paulo'],
    ['Bruno', 35, 'Rio de Janeiro'],
    ['Carla', 22, 'Belo Horizonte']
]

with open('pessoas.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(dados)

print('CSV criado!')


In [ ]:
# Lendo com csv.reader (cada linha vira uma lista)
with open('pessoas.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    for linha in reader:
        print(linha)


In [ ]:
# DictReader: cada linha vira um dicionário (mais legível)
with open('pessoas.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for linha in reader:
        print(linha['nome'], '-', linha['cidade'])


In [ ]:
# Lendo com pandas (mais prático)
df = pd.read_csv('pessoas.csv')
print("Tabela:")
display(df)


In [ ]:
# Operações comuns
print("Informações:")
df.info()

print("\nEstatísticas da idade:")
display(df['idade'].describe())

print("\nPessoas com mais de 25 anos:")
display(df[df['idade'] > 25])


In [ ]:
# Nova coluna calculada + salvar
df['idade_em_5_anos'] = df['idade'] + 5
df.to_csv('pessoas_atualizado.csv', index=False)

display(df)
print("\nArquivo 'pessoas_atualizado.csv' salvo!")


## 4. JSON (JavaScript Object Notation)

JSON é ideal para dados com **estrutura hierárquica**. Principais funções:

| Função | O que faz |
|--------|-----------|
| `json.dump(dados, arquivo)` | Salva em arquivo |
| `json.load(arquivo)` | Lê de arquivo |
| `json.dumps(dados)` | Converte para string |
| `json.loads(texto)` | Converte string para Python |


In [ ]:
# Criando e salvando JSON
dados_json = {
    'curso': 'Python para Análise de Dados',
    'alunos': [
        {'nome': 'Ana', 'nota': 9.5},
        {'nome': 'Bruno', 'nota': 7.8}
    ],
    'ativo': True
}

with open('curso.json', 'w', encoding='utf-8') as f:
    json.dump(dados_json, f, indent=4, ensure_ascii=False)

print('JSON salvo!')


In [ ]:
# Lendo JSON
with open('curso.json', 'r', encoding='utf-8') as f:
    dados_lidos = json.load(f)

print(dados_lidos)
print('Nome do curso:', dados_lidos['curso'])
print('Primeiro aluno:', dados_lidos['alunos'][0]['nome'])


In [ ]:
# JSON → DataFrame (muito útil!)
df_alunos = pd.DataFrame(dados_lidos['alunos'])
display(df_alunos)


In [ ]:
# dumps/loads: convertendo entre dict e string (sem arquivo)
texto_json = json.dumps(dados_json, indent=2, ensure_ascii=False)
print(texto_json)
print("\nTipo da string:", type(texto_json))

objeto = json.loads(texto_json)
print("\nDe volta a dict:", type(objeto))
print(objeto['curso'])


## 5. Excel (.xlsx)

Usamos `pandas` + `openpyxl`. Funções: `pd.read_excel()` e `df.to_excel()`.


In [ ]:
# Criando e salvando Excel
df_vendas = pd.DataFrame({
    'produto': ['Notebook', 'Mouse', 'Teclado', 'Monitor'],
    'preco': [3500, 80, 150, 900],
    'quantidade': [10, 50, 30, 15]
})

df_vendas.to_excel('vendas.xlsx', index=False, sheet_name='Vendas')
print('Excel salvo!')


In [ ]:
# Lendo e calculando
df_lido = pd.read_excel('vendas.xlsx', sheet_name='Vendas')
df_lido['total'] = df_lido['preco'] * df_lido['quantidade']
display(df_lido)


In [ ]:
# Múltiplas abas no mesmo arquivo
df_clientes = pd.DataFrame({
    'cliente': ['Loja A', 'Loja B'],
    'cidade': ['Curitiba', 'Salvador']
})

with pd.ExcelWriter('relatorio.xlsx') as writer:
    df_lido.to_excel(writer, sheet_name='Vendas', index=False)
    df_clientes.to_excel(writer, sheet_name='Clientes', index=False)

print('Relatório com múltiplas abas salvo!')


## 6. XML (eXtensible Markup Language)

XML organiza dados em **tags hierárquicas**. Usamos `xml.etree.ElementTree` (já vem com Python).


In [ ]:
# Criando XML
root = ET.Element('escola')
root.set('nome', 'Escola ABC')

aluno1 = ET.SubElement(root, 'aluno')
ET.SubElement(aluno1, 'nome').text = 'Ana Silva'
ET.SubElement(aluno1, 'nota').text = '9.5'

aluno2 = ET.SubElement(root, 'aluno')
ET.SubElement(aluno2, 'nome').text = 'Bruno Costa'
ET.SubElement(aluno2, 'nota').text = '8.2'

tree = ET.ElementTree(root)
tree.write('escola.xml', encoding='utf-8', xml_declaration=True)
print('XML criado!')


In [ ]:
# Lendo XML
tree = ET.parse('escola.xml')
root = tree.getroot()

print(f'Escola: {root.get("nome")}')
print('\nAlunos:')
for aluno in root.findall('aluno'):
    nome = aluno.find('nome').text
    nota = aluno.find('nota').text
    print(f'  {nome}: {nota}')


In [ ]:
# XML → DataFrame
dados = []
for aluno in root.findall('aluno'):
    dados.append({
        'nome': aluno.find('nome').text,
        'nota': float(aluno.find('nota').text)
    })

df_alunos_xml = pd.DataFrame(dados)
display(df_alunos_xml)


In [ ]:
# Modificando e salvando XML
for aluno in root.findall('aluno'):
    if aluno.find('nome').text == 'Ana Silva':
        aluno.find('nota').text = '9.8'

tree = ET.ElementTree(root)
tree.write('escola.xml', encoding='utf-8', xml_declaration=True)
print('XML atualizado!')


## 7. PDF (Portable Document Format)

PDF preserva layout visual. Principais bibliotecas:
- **pdfplumber** — extrair texto e tabelas
- **reportlab** — criar PDFs do zero

Vamos criar um PDF com reportlab e depois lê-lo com pdfplumber.


In [ ]:
# Criando um PDF com reportlab
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

c = canvas.Canvas('relatorio_novo.pdf', pagesize=letter)
width, height = letter

c.setFont('Helvetica-Bold', 16)
c.drawString(50, height - 50, 'Relatorio de Vendas')

c.setFont('Helvetica', 12)
c.drawString(50, height - 80, 'Empresa: Venda ABC Ltda')
c.drawString(50, height - 100, 'Periodo: Janeiro 2024')
c.drawString(50, height - 120, 'Total de vendas: R$ 150.000,00')

c.save()
print('PDF criado!')


In [ ]:
# Lendo texto do PDF com pdfplumber
import pdfplumber

with pdfplumber.open('relatorio_novo.pdf') as pdf:
    print(f'Total de paginas: {len(pdf.pages)}')
    primeira_pagina = pdf.pages[0]
    texto = primeira_pagina.extract_text()
    print(texto)


In [ ]:
# Criando PDF com tabela
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

dados_tabela = [
    ['Produto', 'Quantidade', 'Preco'],
    ['Notebook', '10', 'R$ 3.500,00'],
    ['Mouse', '50', 'R$ 80,00'],
    ['Teclado', '30', 'R$ 150,00']
]

doc = SimpleDocTemplate('tabela_vendas.pdf', pagesize=letter)
story = []

styles = getSampleStyleSheet()
titulo = Paragraph('Relatorio de Vendas', styles['Heading1'])
story.append(titulo)

tabela = Table(dados_tabela)
tabela.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('GRID', (0, 0), (-1, -1), 1, colors.black)
]))
story.append(tabela)

doc.build(story)
print('PDF com tabela criado!')

# Extraindo tabela do PDF
with pdfplumber.open('tabela_vendas.pdf') as pdf:
    tabelas = pdf.pages[0].extract_tables()
    if tabelas:
        df_pdf = pd.DataFrame(tabelas[0][1:], columns=tabelas[0][0])
        display(df_pdf)


## 8. DOCX (Microsoft Word Document)

DOCX é o formato de documentos editáveis do Word. Usamos a biblioteca **python-docx**.


In [ ]:
# Criando um DOCX do zero
from docx import Document
from docx.shared import Pt, RGBColor

doc = Document()
doc.add_heading('Relatorio de Vendas', 0)
doc.add_paragraph('Este e um relatorio de vendas gerado automaticamente.')

p = doc.add_paragraph('Periodo: Janeiro 2024')
p.runs[0].bold = True
p.runs[0].font.size = Pt(14)

# Adicionando tabela
tabela = doc.add_table(rows=4, cols=3)
tabela.style = 'Light Grid Accent 1'

celulas = tabela.rows[0].cells
celulas[0].text = 'Produto'
celulas[1].text = 'Quantidade'
celulas[2].text = 'Preco'

dados = [
    ['Notebook', '10', 'R$ 3.500,00'],
    ['Mouse', '50', 'R$ 80,00'],
    ['Teclado', '30', 'R$ 150,00']
]

for i, linha in enumerate(dados, 1):
    celulas = tabela.rows[i].cells
    celulas[0].text = linha[0]
    celulas[1].text = linha[1]
    celulas[2].text = linha[2]

doc.save('relatorio_novo.docx')
print('DOCX criado!')


In [ ]:
# Lendo DOCX
doc = Document('relatorio_novo.docx')
print('Conteudo do documento:')
for paragrafo in doc.paragraphs:
    if paragrafo.text.strip():
        print(paragrafo.text)


In [ ]:
# Extraindo tabela do DOCX para DataFrame
if doc.tables:
    tabela = doc.tables[0]
    dados = []
    for linha in tabela.rows:
        dados.append([celula.text for celula in linha.cells])

    df_docx = pd.DataFrame(dados[1:], columns=dados[0])
    display(df_docx)


In [ ]:
# DOCX com formatação avançada
from docx.enum.text import WD_ALIGN_PARAGRAPH

doc2 = Document()

titulo = doc2.add_heading('Relatorio Executivo', 0)
titulo.paragraph_format.alignment = WD_ALIGN_PARAGRAPH.CENTER

doc2.add_paragraph(
    'Este documento foi gerado automaticamente.',
    style='List Bullet'
)

p = doc2.add_paragraph()
p.add_run('Negrito: ').bold = True
p.add_run('Este e um texto em negrito. ')
p.add_run('Italico: ').italic = True
p.add_run('Este e um texto em italico.')

p_vermelho = doc2.add_paragraph('Texto em cor vermelha')
for run in p_vermelho.runs:
    run.font.color.rgb = RGBColor(255, 0, 0)

doc2.save('relatorio_formatado.docx')
print('DOCX com formatacao criado!')


## 9. Banco de dados SQLite

SQLite é um banco em **arquivo único** — não precisa de servidor. Perfeito para aprendizado e prototipagem.

Fluxo: `connect()` → `cursor()` → `execute()` → `commit()` → `close()`


In [ ]:
# Criando tabela
conexao = sqlite3.connect('banco.db')
cursor = conexao.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS funcionarios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    cargo TEXT,
    salario REAL
)
''')

conexao.commit()
print('Tabela criada!')


In [ ]:
# Inserindo registros
funcionarios = [
    ('Mariana', 'Analista', 5000),
    ('Pedro', 'Gerente', 9000),
    ('Joana', 'Desenvolvedora', 7000)
]

cursor.executemany(
    'INSERT INTO funcionarios (nome, cargo, salario) VALUES (?, ?, ?)',
    funcionarios
)
conexao.commit()
print(f'{cursor.rowcount} registros inseridos!')


In [ ]:
# Consultando com SQL
cursor.execute('SELECT * FROM funcionarios WHERE salario > 6000')
resultados = cursor.fetchall()

for linha in resultados:
    print(linha)


In [ ]:
# SQL → DataFrame diretamente
df_func = pd.read_sql_query('SELECT * FROM funcionarios', conexao)
display(df_func)


In [ ]:
# Fechando conexão (importante para liberar o arquivo)
conexao.close()
print('Conexao encerrada')


## 10. Consumindo dados de uma API

APIs devolvem dados (geralmente JSON) via requisições HTTP. Usamos a biblioteca `requests`.

Fluxo: `requests.get(url)` → `resposta.json()` → `pd.json_normalize()`


In [ ]:
# Fazendo requisição GET
resposta = requests.get('https://jsonplaceholder.typicode.com/users')

print('Status code:', resposta.status_code)

if resposta.status_code == 200:
    dados_api = resposta.json()
    print(f'{len(dados_api)} usuarios retornados')
    print('\nPrimeiro usuario:')
    print(dados_api[0])


In [ ]:
# JSON → DataFrame (json_normalize achata estruturas aninhadas)
df_usuarios = pd.json_normalize(dados_api)
display(df_usuarios[['id', 'name', 'email', 'address.city']])


In [ ]:
# Salvando dados da API em CSV
df_usuarios.to_csv('usuarios_api.csv', index=False)
print('Dados da API salvos em CSV!')


## 11. Upload e Download no Google Colab

> Esta seção funciona **apenas no Google Colab**. Se estiver executando localmente, pule para a seção seguinte.

O Colab roda em uma máquina remota sem acesso aos seus arquivos locais.
Use `google.colab.files` para transferir arquivos.


In [ ]:
# Upload: enviar arquivo do computador para o Colab
from google.colab import files

uploaded = files.upload()

for nome_arquivo in uploaded.keys():
    print(f'Arquivo "{nome_arquivo}" enviado, {len(uploaded[nome_arquivo])} bytes')

# Após o upload, o arquivo está disponível para leitura com pandas
# df = pd.read_csv(nome_arquivo)


In [ ]:
# Download: baixar arquivo do Colab para o computador
from google.colab import files

# files.download('relatorio.xlsx')
print('Descomente a linha acima com o nome do arquivo que deseja baixar.')


## 12. Conectando ao Google Drive

> Esta seção funciona **apenas no Google Colab**.

Arquivos criados no Colab são temporários. Montar o Drive permite salvar permanentemente.


In [ ]:
# Montando o Google Drive
from google.colab import drive

drive.mount('/content/drive')
# Uma janela pedirá autorização na primeira execução


In [ ]:
# Salvando no Drive
caminho_drive = '/content/drive/My Drive/pessoas_drive.csv'

df.to_csv(caminho_drive, index=False)
print('Arquivo salvo no Google Drive!')


In [ ]:
# Lendo do Drive
df_do_drive = pd.read_csv(caminho_drive)
display(df_do_drive)


## 13. Resumo: qual formato usar em cada situação

| Formato | Quando usar |
|---------|-------------|
| TXT | Logs, anotações simples, textos corridos |
| CSV | Tabelas simples, troca de dados entre sistemas |
| JSON | Dados hierárquicos, configurações, respostas de APIs |
| Excel | Relatórios para humanos, múltiplas abas, formatação visual |
| XML | Configurações, troca de dados estruturados |
| PDF | Relatórios finalizados, documentos nao-editaveis |
| DOCX | Documentos editaveis, relatorios formatados |
| SQLite | Dados relacionais, consultas complexas |
| API + requests | Dados em tempo real, integracao externa |


## 14. Exercícios Práticos

> As soluções estão no arquivo `fontes_de_dados_solucoes.ipynb`.
> Tente resolver sozinho antes de consultar as respostas!


### Exercício 1: CSV — Cálculo de Estoque

Crie um arquivo CSV com uma lista de produtos (nome, preço, quantidade em estoque).
        Usando pandas, calcule o valor total do estoque (preço × quantidade) e salve o resultado em um novo CSV.


In [ ]:
# Escreva sua solucao para Exercício 1


### Exercício 2: JSON + API

Faça uma requisição para `https://jsonplaceholder.typicode.com/posts`,
        transforme o resultado em DataFrame e salve apenas as colunas `userId` e `title` em um arquivo Excel.


In [ ]:
# Escreva sua solucao para Exercício 2


### Exercício 3: SQLite — JOIN entre tabelas

Crie um banco com duas tabelas: `clientes` (id, nome, cidade) e `pedidos` (id, cliente_id, valor).
        Insira registros e faça uma consulta SQL que junte (JOIN) as duas tabelas, mostrando nome do cliente com seus pedidos.


In [ ]:
# Escreva sua solucao para Exercício 3


### Exercício 4: XML

Crie um arquivo XML com uma lista de livros (título, autor, ano).
        Depois leia o arquivo e exporte os dados para um DataFrame.


In [ ]:
# Escreva sua solucao para Exercício 4


### Exercício 5: PDF com Tabela

Crie um DataFrame com dados de vendas, gere um PDF com tabela usando reportlab,
        depois use pdfplumber para extrair os dados de volta para um DataFrame e compare os dois.


In [ ]:
# Escreva sua solucao para Exercício 5


### Exercício 6: DOCX

Crie um documento Word com título, parágrafos formatados e uma tabela contendo dados de um DataFrame.


In [ ]:
# Escreva sua solucao para Exercício 6


### Exercício 7: Upload + Google Drive (Colab)

Faça upload de um arquivo CSV, use pandas para remover linhas com valores nulos (`df.dropna()`),
        e salve o resultado limpo no Google Drive.


In [ ]:
# Escreva sua solucao para Exercício 7


### Exercício 8: Desafio Extra

Combine tudo! Busque dados de uma API, salve-os em SQLite, depois leia do SQLite com pandas,
        crie um documento DOCX com tabelas, exporte para PDF com gráficos, e finalize com um arquivo Excel resumido.


In [ ]:
# Escreva sua solucao para Exercício 8
